# Chapter 4 - Improving Embeddings with Biased Random Walks in Node2Vec

Hands-On Graph Neural Networks Using Python, Second Edition

Runs the code of the chapter in order and prints the outputs shown in the book.
Saves Figures 4.4 and 4.5 to figures/. Downloads MovieLens 100K on first run.
Biased walks are generated by the chapter's own random_walk function.

Usage:
    python run.py

In [ ]:
import os
import random
import sys
from collections import defaultdict
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import Node2Vec
from torch_geometric.utils import from_networkx, to_networkx

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                '..', 'tools'))
from bookstyle import apply, figure, FONT, BG, DPI, INK, G3, G4, G5, G6

plt = apply()

FIG_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# The two factions have to be told apart in a grayscale print, so they differ
# by two steps of the ramp and both carry dark labels. A reversed white label
# on a dark node closes up at this point size once the ink spreads.
CLASS_A, CLASS_B = G4, G6

# The heatmap ramp stops at G4 instead of running to black. A darker cell would
# need its numerals reversed out in white, and at 8.5 pt the press closes up
# the counters of reversed type; G4 still carries INK at 6.6:1.
CELLS = LinearSegmentedColormap.from_list("book_greys", [BG, G6, G5, G4])


def save(fig, name):
    """Like bookstyle.save, but shows the figure inline before closing it."""
    fig.savefig(os.path.join(FIG_DIR, name), dpi=DPI, facecolor=BG,
                edgecolor="none")
    print(f"[saved figures/{name}]")
    if "ipykernel" in sys.modules:
        plt.show()
    else:
        plt.close(fig)


def banner(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

## PART 1 - Biased random walks

In [ ]:
banner("PART 1 - Biased random walks")

random.seed(0)
np.random.seed(0)

G = nx.erdos_renyi_graph(10, 0.3, seed=1, directed=False)


def next_node(G, previous, current, p, q):
    neighbors = list(G.neighbors(current))
    alphas = []
    for neighbor in neighbors:
        if neighbor == previous:
            alpha = 1 / p
        elif previous is not None and G.has_edge(neighbor, previous):
            alpha = 1
        else:
            alpha = 1 / q
        alphas.append(alpha * G[current][neighbor].get('weight', 1))
    probs = [a / sum(alphas) for a in alphas]
    next_node_id = np.random.choice(neighbors, size=1, p=probs)[0]
    return next_node_id


def random_walk(G, start, length, p, q):
    walk = [start]
    for _ in range(length):
        current = walk[-1]
        previous = walk[-2] if len(walk) > 1 else None
        neighbors = list(G.neighbors(current))
        if not neighbors:
            break
        walk.append(next_node(G, previous, current, p, q))
    return [str(x) for x in walk]


print(random_walk(G, start=0, length=8, p=1, q=1))
print(random_walk(G, start=0, length=8, p=1, q=10))
print(random_walk(G, start=0, length=8, p=10, q=1))

## PART 2 - Zachary's Karate Club

In [ ]:
banner("PART 2 - Zachary's Karate Club")

dataset = KarateClub()
data = dataset[0]

G_nx = nx.karate_club_graph()
labels = np.array([1 if G_nx.nodes[n]['club'] == 'Officer' else 0
                   for n in G_nx.nodes])

G = to_networkx(data, to_undirected=True)
pos = nx.spring_layout(G, seed=0)
node_colors = [CLASS_A if l == 0 else CLASS_B for l in labels]

fig, ax = figure(aspect=0.84)
ax.axis("off")
# 32 and 33 are 0.25in apart once the layout is stretched to the column, so a
# node wider than that runs its neighbour's label into its own.
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=G6, width=0.7)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=200, node_color=node_colors,
                       edgecolors=INK, linewidths=0.7)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color=INK,
                        labels={n: str(n) for n in G.nodes},
                        font_weight="bold", font_family=FONT)
# networkx fits the data exactly, which clips the outermost nodes in half.
ax.margins(0.09)
ax.legend(handles=[mpatches.Patch(facecolor=CLASS_A, edgecolor=G3,
                                  label="Mr. Hi's faction"),
                   mpatches.Patch(facecolor=CLASS_B, edgecolor=G3,
                                  label="Officer's faction")],
          loc="upper left", fontsize=8, framealpha=1.0, edgecolor=G6)
save(fig, "fig4_4_karate_club.png")

## PART 3 - Training Node2Vec

In [ ]:
banner("PART 3 - Training Node2Vec")

def train_node2vec(G, p, q, walks_per_node=80, length=10, context_size=10,
                   dim=128, epochs=5, seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)

    walks = [random_walk(G, node, length, p, q)
             for _ in range(walks_per_node) for node in G.nodes]
    node_idx = {str(node): i for i, node in enumerate(G.nodes)}
    walks = torch.tensor([[node_idx[n] for n in walk] for walk in walks])
    pos_rw = walks.unfold(1, context_size, 1).reshape(-1, context_size)

    model = Node2Vec(from_networkx(G).edge_index, embedding_dim=dim,
                     walk_length=length, context_size=context_size,
                     sparse=True)
    with torch.no_grad():
        model.embedding.weight.uniform_(-0.5 / dim, 0.5 / dim)
    optimizer = torch.optim.SparseAdam(model.parameters(), lr=0.01)

    num_nodes = G.number_of_nodes()
    for _ in range(epochs):
        for batch in torch.randperm(pos_rw.size(0)).split(512):
            pos = pos_rw[batch]
            neg = torch.cat([
                pos[:, :1].repeat(5, 1),
                torch.randint(num_nodes, (pos.size(0) * 5, context_size - 1)),
            ], dim=1)
            optimizer.zero_grad()
            loss = model.loss(pos, neg)
            loss.backward()
            optimizer.step()

    return model().detach().numpy()


def evaluate(embeddings, labels, n_runs=10):
    accs = []
    for seed in range(n_runs):
        X_tr, X_te, y_tr, y_te = train_test_split(
            embeddings, labels,
            test_size=0.4, stratify=labels, random_state=seed
        )
        clf = RandomForestClassifier(n_estimators=100, random_state=seed)
        clf.fit(X_tr, y_tr)
        accs.append(accuracy_score(y_te, clf.predict(X_te)))
    return np.mean(accs), np.std(accs)

## PART 4 - DeepWalk versus Node2Vec

In [ ]:
banner("PART 4 - DeepWalk versus Node2Vec")

emb_deepwalk = train_node2vec(G, p=1, q=1)
emb_node2vec = train_node2vec(G, p=1, q=0.25)

mean_dw, std_dw = evaluate(emb_deepwalk, labels)
mean_n2v, std_n2v = evaluate(emb_node2vec, labels)

print(f'DeepWalk (p=1, q=1): {mean_dw*100:.2f}% ± {std_dw*100:.2f}%')
print(f'Node2Vec (p=1, q=0.25): {mean_n2v*100:.2f}% ± {std_n2v*100:.2f}%')

## PART 5 - Parameter search

In [ ]:
banner("PART 5 - Parameter search")

p_values = [0.25, 0.5, 1, 2, 4]
q_values = [0.25, 0.5, 1, 2, 4]
results = np.zeros((len(p_values), len(q_values)))

for i, p in enumerate(p_values):
    for j, q in enumerate(q_values):
        emb = train_node2vec(G, p=p, q=q)
        results[i, j], _ = evaluate(emb, labels)

print("q:      " + "  ".join(f"{q:>5}" for q in q_values))
for i, p in enumerate(p_values):
    print(f"p={p:<5} " + "  ".join(f"{v:.3f}" for v in results[i]))

fig, ax = figure(aspect=0.62)
im = ax.imshow(results, cmap=CELLS, vmin=0.90, vmax=1.00,
               aspect="auto", origin="upper")

for i in range(len(p_values)):
    for j in range(len(q_values)):
        ax.text(j, i, f"{results[i, j]:.3f}", ha="center", va="center",
                fontsize=8.5, fontweight="bold", fontfamily=FONT, color=INK)

ax.set_xticks(range(len(q_values)))
ax.set_yticks(range(len(p_values)))
ax.set_xticklabels([f"q={q}" for q in q_values], fontsize=8.5, fontfamily=FONT)
ax.set_yticklabels([f"p={p}" for p in p_values], fontsize=8.5, fontfamily=FONT)
# Each parameter name is split over two lines rather than set as one long
# string: the y label is as tall as the whole figure and would run out of room.
ax.set_xlabel("In-out parameter q\n(lower → more exploration, DFS-like)",
              fontsize=8.5, color=INK, fontfamily=FONT, labelpad=5)
ax.set_ylabel("Return parameter p\n(higher → less backtracking)",
              fontsize=8.5, color=INK, fontfamily=FONT, labelpad=5)
# The caption names the figure and says which cell is outlined, so the title
# carries only what the caption leaves out: where the numbers come from.
ax.set_title("Zachary's Karate Club, 10 splits, 60/40 train/test",
             fontsize=8.5, color=INK, fontfamily=FONT, pad=5)
bar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
bar.set_label("Accuracy", fontsize=8.5, fontfamily=FONT, color=INK)
bar.ax.tick_params(labelsize=8)
di, dj = p_values.index(1), q_values.index(1)
ax.add_patch(Rectangle((dj - 0.5, di - 0.5), 1, 1, linewidth=1.6,
                       edgecolor=INK, facecolor="none"))
save(fig, "fig4_5_pq_heatmap.png")

## PART 6 - Building the movie graph

In [ ]:
banner("PART 6 - Building the movie graph")

if not os.path.isdir('ml-100k'):
    url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
    with urlopen(url) as zurl:
        with ZipFile(BytesIO(zurl.read())) as zfile:
            zfile.extractall('.')

ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t',
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)
movies = pd.read_csv(
    'ml-100k/u.item', sep='|',
    usecols=range(2),
    names=['movie_id', 'title'],
    encoding='latin-1'
)

print(ratings.head(3))
print(f'\n{len(ratings)} ratings, {movies.shape[0]} movies')

ratings = ratings[ratings['rating'] >= 4]
print(f'{len(ratings)} positive ratings remaining')

pairs = defaultdict(int)
for _, group in ratings.groupby('user_id'):
    user_movies = list(group['movie_id'])
    for i in range(len(user_movies)):
        for j in range(i + 1, len(user_movies)):
            pairs[(user_movies[i], user_movies[j])] += 1

G_movies = nx.Graph()
for (movie1, movie2), score in pairs.items():
    if score >= 20:
        G_movies.add_edge(movie1, movie2, weight=score)

print(f'Graph: {G_movies.number_of_nodes()} movies, '
      f'{G_movies.number_of_edges()} edges')

## PART 7 - Training Node2Vec on the movie graph

In [ ]:
banner("PART 7 - Training Node2Vec on the movie graph")

node_list = list(G_movies.nodes())
idx_to_movie = {i: m for i, m in enumerate(node_list)}
movie_to_idx = {m: i for i, m in idx_to_movie.items()}

movie_emb = train_node2vec(G_movies, p=2, q=1, walks_per_node=10,
                           length=20, dim=64)
print(f'Embeddings: {movie_emb.shape}')

## PART 8 - Making recommendations

In [ ]:
banner("PART 8 - Making recommendations")

emb_norm = F.normalize(torch.tensor(movie_emb), dim=1)


def recommend(title, top_k=5):
    row = movies[movies['title'] == title]
    if row.empty:
        print(f'Movie not found: {title}')
        return
    movie_id = row['movie_id'].values[0]
    if movie_id not in movie_to_idx:
        print(f'{title} not in graph (too few co-likes)')
        return
    idx = movie_to_idx[movie_id]
    sims = (emb_norm @ emb_norm[idx]).numpy()
    top = np.argsort(-sims)[1:top_k+1]
    print(f'Movies most similar to "{title}":')
    for rank, i in enumerate(top, 1):
        mid = idx_to_movie[i]
        mtitle = movies[movies['movie_id'] == mid]['title'].values[0]
        print(f'   {rank}. {mtitle} (similarity: {sims[i]:.2f})')


recommend('Star Wars (1977)')